# add-sub-div-back-lambdas — worked example 3: Reverse pass for z = (a + b) / c via BACK dispatch

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `add-sub-div-back-lambdas`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A reverse pass walks ops in reverse topological order, calling each op's backward lambda to route the upstream gradient to its inputs. For `s = a + b; z = s / c`, we first dispatch `div` (arg0 gives `grad_s`, arg1 gives `grad_c`), then dispatch `add` on `grad_s` to split it to `a` and `b`. Because add copies its gradient, `grad_a == grad_b == grad_s`.

## Worked solution

**Goal.** Implement a dispatcher `mini_back` plus `reverse_add_div_chain(a, b, c)` returning `{'da','db','dc'}` for `z = (a + b) / c`, with `dL/dz` seeded to ones.

**Step 1 — dispatcher.** `mini_back(op, arg, g, o, x, y)` simply does `BACK[(op, arg)](g, o, x, y)`. It never names a specific backward; it only indexes the dict. This is the dispatch pattern an autograd engine uses at reverse time.

**Step 2 — forward, caching intermediates.** Compute `s = a + b` then `z = s / c`. We need `s`, `c`, and `z` saved because the div backwards reference them.

**Step 3 — seed and walk div.** Seed `grad_z = ones_like(z)`. The last op is the div, whose inputs are `(s, c)`. Dispatch arg0 to get `grad_s = grad_z / c` and arg1 to get `grad_c = -grad_z * s / c**2`.

**Step 4 — walk add.** The add's inputs are `(a, b)`; its upstream gradient is `grad_s`. Dispatch arg0 and arg1; both return `grad_s` unchanged, so `grad_a = grad_b = grad_s = 1/c`.

**Step 5 — verify.** Closed form: `dL/da = dL/db = 1/c` and `dL/dc = -(a+b)/c**2`. We confirm against autograd.

In [ ]:
BACK = {
    ('add', 0): lambda g, o, x, y: g,
    ('add', 1): lambda g, o, x, y: g,
    ('div', 0): lambda g, o, x, y: g / y,
    ('div', 1): lambda g, o, x, y: -g * x / (y * y),
}

def mini_back(op_name, argnum, grad_out, out, x, y):
    return BACK[(op_name, argnum)](grad_out, out, x, y)

def reverse_add_div_chain(a, b, c):
    s = a + b
    z = s / c
    grad_z = t.ones_like(z)
    grad_s = mini_back('div', 0, grad_z, z, s, c)
    grad_c = mini_back('div', 1, grad_z, z, s, c)
    grad_a = mini_back('add', 0, grad_s, s, a, b)
    grad_b = mini_back('add', 1, grad_s, s, a, b)
    return {'da': grad_a, 'db': grad_b, 'dc': grad_c}

t.manual_seed(0)
a = t.randn(5, requires_grad=True)
b = t.randn(5, requires_grad=True)
c = (t.randn(5).abs() + 0.5).requires_grad_(True)
out = (a + b) / c
out.sum().backward()

grads = reverse_add_div_chain(a.detach(), b.detach(), c.detach())
print('da match:', t.allclose(grads['da'], a.grad, atol=1e-5))
print('db match:', t.allclose(grads['db'], b.grad, atol=1e-5))
print('dc match:', t.allclose(grads['dc'], c.grad, atol=1e-5))